# Load Data

NOTE: before running this section of the code, you will need to add the `Frozone_Data` folder to `data/experiment_results`. This folder should be exactly as provided by Laur in the shared drive.

In [ ]:
import pandas as pd
from pathlib import Path

repo_root = Path("__file__").resolve().parent.parent.parent
data_path = repo_root / "data/experiment_results/Frozone_Data/survey2/4-14_survey2_values.csv"
results_data = pd.read_csv(data_path, header=1, skiprows=[2])

In [ ]:
results_data.head()

# Data Vis

You may have to install matplotlib and seaborn to run the cells below.

## Violin Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_field_violins(df, category_template):
    """
    Creates violin plots for CName, FName, and HName for a given question category.

    Parameters:
        df: DataFrame loaded from the values CSV (numeric 1-5 responses)
        category_template: Question string with [Field-Name] as placeholder,
                           e.g. "I felt [Field-Name] responded in a way that
                                addressed or countered misrepresentations."
    """
    col_map = {
        "Coolbot": category_template.replace("[Field-Name]", "[Field-CName]"),
        "Frobot": category_template.replace("[Field-Name]", "[Field-FName]"),
        "Hotbot": category_template.replace("[Field-Name]", "[Field-HName]"),
    }

    missing = [col for col in col_map.values() if col not in df.columns]
    if missing:
        raise ValueError(f"Columns not found in DataFrame:\n" + "\n".join(missing))

    plot_df = pd.concat([
        df[col].dropna().rename("Response").to_frame().assign(Field=label)
        for label, col in col_map.items()
    ])

    order = ["Frobot", "Coolbot", "Hotbot"]

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.violinplot(data=plot_df, x="Field", y="Response", inner="box",
                   order=order, ax=ax)

    for i, label in enumerate(order):
        mean_val = plot_df[plot_df["Field"] == label]["Response"].mean()
        ax.scatter(i, mean_val, color="white", edgecolor="black", zorder=5, s=60,
                   label=f"{label} mean: {mean_val:.2f}")

    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_yticklabels(["1 (Strongly Disagree)", "2", "3 (Neutral)", "4", "5 (Strongly Agree)"])
    ax.set_ylim(0.5, 5.5)
    ax.set_title(category_template.replace("[Field-Name]", "each participant"), wrap=True)
    ax.set_xlabel("Chatbot")
    ax.set_ylabel("Response")
    ax.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_field_violins(results_data, "I felt [Field-Name] responded in a way that addressed or countered misrepresentations.")


In [ ]:
VALID_VIOLIN_CATEGORIES = [
    "I felt heard and understood by [Field-Name].",
    "I gained a better understanding of [Field-Name]'s perspective.",
    "I feel [Field-Name]'s responses were productive and encouraged frank discussion.",
    "[Field-Name] used toxic or inflammatory language.",
    "I felt [Field-Name] responded in a way that addressed and countered toxic or inflammatory language.",
    "[Field-Name] shared information I believe was factually incorrect.",
    "I felt [Field-Name] responded in a way that addressed and countered information I believe was incorrect.",
    "[Field-Name] made biased comments towards or against groups of people (concerning race, gender, class, sexuality, or something else).",
    "I felt [Field-Name] responded in a way that addressed and countered bias.",
    "[Field-Name]'s reasoning was sometimes flawed.",
    "I felt [Field-Name] responded in a way that addressed flawed reasoning in others' responses.",
    "[Field-Name] misrepresented what I said or what others said.",
    "I felt [Field-Name] responded in a way that addressed or countered misrepresentations.",
]


### Violin Plots across all numerical categories

In [ ]:
for category in VALID_VIOLIN_CATEGORIES:
    plot_field_violins(results_data, category)


## Overall Heatmap

In [ ]:
SHORT_LABELS = [
    "Felt heard",
    "Better understanding of perspective",
    "Responses productive",
    "Used toxic language",
    "Countered toxic language",
    "Shared incorrect info",
    "Countered incorrect info",
    "Made biased comments",
    "Countered bias",
    "Reasoning was flawed",
    "Addressed flawed reasoning",
    "Misrepresented others",
    "Countered misrepresentations",
]

heatmap_data = {}
for label, category in zip(SHORT_LABELS, VALID_VIOLIN_CATEGORIES):
    heatmap_data[label] = {
        "Frobot":  results_data[category.replace("[Field-Name]", "[Field-FName]")].mean(),
        "Coolbot": results_data[category.replace("[Field-Name]", "[Field-CName]")].mean(),
        "Hotbot":  results_data[category.replace("[Field-Name]", "[Field-HName]")].mean(),
    }

heatmap_df = pd.DataFrame(heatmap_data).T[["Frobot", "Coolbot", "Hotbot"]]

fig, ax = plt.subplots(figsize=(7, 9))
sns.heatmap(
    heatmap_df,
    annot=True, fmt=".2f",
    vmin=1, vmax=5,
    cmap="RdYlGn",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Mean Response (1=Strongly Disagree, 5=Strongly Agree)"},
)
ax.set_title("Mean Survey Responses by Chatbot and Category")
ax.set_xlabel("Chatbot")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
NEGATIVE_CATEGORIES = {
    "[Field-Name] used toxic or inflammatory language.",
    "[Field-Name] shared information I believe was factually incorrect.",
    "[Field-Name] made biased comments towards or against groups of people (concerning race, gender, class, sexuality, or something else).",
    "[Field-Name]'s reasoning was sometimes flawed.",
    "[Field-Name] misrepresented what I said or what others said.",
}

heatmap_data_normed = {}
for label, category in zip(SHORT_LABELS, VALID_VIOLIN_CATEGORIES):
    row = {}
    for bot, field in [("Frobot", "FName"), ("Coolbot", "CName"), ("Hotbot", "HName")]:
        col = category.replace("[Field-Name]", f"[Field-{field}]")
        mean = results_data[col].mean()
        if category in NEGATIVE_CATEGORIES:
            mean = 6 - mean  # flip: 1=negative, 5=positive
        row[bot] = mean
    heatmap_data_normed[label] = row

heatmap_df_normed = pd.DataFrame(heatmap_data_normed).T[["Frobot", "Coolbot", "Hotbot"]]

fig, ax = plt.subplots(figsize=(7, 9))
sns.heatmap(
    heatmap_df_normed,
    annot=True, fmt=".2f",
    vmin=1, vmax=5,
    cmap="RdYlGn",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Mean Response (1=Felt Most Negative, 5=Felt Most Positive)"},
)
ax.set_title("Normalized Mean Survey Responses by Chatbot and Category\n(scale normalized: 1=negative, 5=positive)")
ax.set_xlabel("Chatbot")
ax.set_ylabel("")
plt.tight_layout()
plt.show()